In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import os
import fiona

# Select random polygons

In [ ]:
random_size = 5000

In [ ]:
in_file = '../../analysis/clean_summarize/out_polys/sentinel_2021_v7_aea_cleaned_no0.gpkg'
if not os.path.isfile(in_file):
    full_gdf = gpd.read_file('../../analysis/clean_summarize/out_polys/sentinel_2021_v7_aea_cleaned.gpkg')
    full_gdf.loc[full_gdf['DN']==1].to_file(in_file)
with fiona.open(in_file) as fiona_features:
    total_features = len(fiona_features)

In [ ]:
np_rng = np.random.default_rng(seed=12)
random_rows = np_rng.integers(0, total_features-1, random_size)

In [ ]:
full_df = gpd.read_file(in_file, engine='pyogrio',use_arrow=True)
full_df = full_df.iloc[random_rows]
full_df['eval'] = 0 
full_df.to_file('./random_subset2.gpkg')

In [ ]:
print(random_rows)

# Analyze after annotation in QGIS

In [ ]:
annotated = gpd.read_file('./random_subset2_annotated.gpkg')
annotated['geometry'] = annotated.make_valid()

In [ ]:
# Overall Accuracy
print((annotated.groupby('eval')['DN'].count()))
print(100*annotated.groupby('eval')['DN'].count()/annotated.shape[0])

In [ ]:
# Accuracy over labeling
expanding_avg = (annotated['eval'] == 0).expanding(1).mean()
expanding_avg.plot(xlabel='Number labeled', ylabel='Precision')

In [ ]:
rolling_avg = (annotated['eval'] == 0).rolling(100,1).mean()
rolling_avg.plot()

In [ ]:
# By size
size_dict = {
    'remove': [0, 0.05],
    'very_small': [0.05, 0.1],
    'small': [0.1, 1],
    'medium': [1, 10],
    'large': [10, 50],
    'xlarge': [50, 1000000],
}

annotated['area_ha'] = annotated.geometry.area/10000
annotated['tp'] = annotated['eval']==0
#assign size dict
annotated['size_class'] = None
for s in size_dict.keys():
    size_filter = (annotated['area_ha'] >= size_dict[s][0]) * (annotated['area_ha'] < size_dict[s][1])
    annotated.loc[size_filter, 'size_class'] = s
print(annotated.groupby(['size_class'])['tp'].agg(['mean', 'count']))

In [ ]:
# By biome
biome_gdf = gpd.read_file('../regions/data/lm_bioma_250.shp')
annotated_centers = annotated.copy()
annotated_centers['geometry'] = annotated.geometry.centroid
ann_biome = annotated_centers.sjoin_nearest(biome_gdf.to_crs('ESRI:102033'))
ann_biome.groupby('Bioma')['tp'].agg(['count','mean'])